In [1]:
from sqlalchemy import Integer, MetaData, String, Column, VARCHAR, DATE, DateTime, Table, ForeignKey
import pandas as pd 
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from db.connection import get_engine

## using the table and meta data directly 

In [2]:
engine = get_engine()

In [3]:
df = pd.read_csv("../data/Cleaned_Data.csv")
engine = get_engine()
meta = MetaData()

# creating the customers table 
customers = Table(
    "customers", 
    meta,
    Column("customer_id", Integer, primary_key=True, autoincrement=True),
    Column("customer_name", VARCHAR(100), nullable=False),
    Column("customer_email", VARCHAR(100), nullable=False),
    Column("customer_age", Integer, nullable=False),
    Column("customer_gender", VARCHAR(20), nullable=False)
)

# creating the tickets table 
tickets = Table(
    "tickets", 
    meta,
    Column("owner", Integer, ForeignKey("customers.customer_id")),
    Column("ticket_id", Integer, nullable=False, primary_key=True),
    Column("ticket_type", VARCHAR(100), nullable=False),
    Column("ticket_subject", VARCHAR(200), nullable=False),
    Column("ticket_description", String, nullable=False),
    Column("ticket_status", VARCHAR(50), nullable=False),
    Column("resolution", String, nullable=True),                
    Column("ticket_priority", VARCHAR(50), nullable=True),
    Column("ticket_channel", VARCHAR(50), nullable=True),
    Column("product_purchased", VARCHAR(100), nullable=False),
    Column("date_of_purchase", DATE, nullable=False),
    Column("first_response_time", DateTime, nullable=True),
    Column("time_to_resolution", DateTime, nullable=True),
    Column("customer_satisfaction_rating", Integer, nullable=True)
)

# reset the engine
tickets.drop(engine, checkfirst=True)
customers.drop(engine, checkfirst=True)
meta.create_all(engine)

# Extract and insert unique customers
unique_customers = df[["customer_name", "customer_email", "customer_age", "customer_gender"]].drop_duplicates(subset=["customer_email"]).copy()
unique_customers.to_sql("customers", engine, if_exists="append", index=False)
print("Customers table inserted!")

# Fetch generated customer IDs from lowercase table
db_customers = pd.read_sql("SELECT customer_id, customer_email FROM customers", engine)

# Merge back onto dataframe and rename customer_id -> owner
tickets_df = df.merge(db_customers, on="customer_email", how="left")
tickets_df = tickets_df.rename(columns={"customer_id": "owner"})

tickets_to_insert = tickets_df[[
    "ticket_id", "owner", "product_purchased", "date_of_purchase",
    "ticket_type", "ticket_subject", "ticket_description", "ticket_status",
    "resolution", "ticket_priority", "ticket_channel", "first_response_time",
    "time_to_resolution", "customer_satisfaction_rating"
]]

tickets_to_insert.to_sql("tickets", engine, if_exists="append", index=False)
print("Tickets table inserted!")

print("Data uploaded successfully!")

Customers table inserted!
Tickets table inserted!
Data uploaded successfully!


In [4]:
customers.primary_key

PrimaryKeyConstraint(Column('customer_id', Integer(), table=<customers>, primary_key=True, nullable=False))

## ORM training 

In [64]:
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session
from sqlalchemy import VARCHAR, ForeignKey, select
from datetime import date,  datetime
from typing import List, Optional
from pathlib import Path
import pandas as pd
import sys
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from db.connection import get_engine
class Base(DeclarativeBase):
    pass

creating the engine

In [58]:
engine = get_engine()

defining the schemas for the database tables

In [59]:
class Customers(Base):
    __tablename__ = "customers"

    customer_id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    customer_name: Mapped[str] = mapped_column(VARCHAR(100), nullable=False)
    customer_email: Mapped[str] = mapped_column(VARCHAR(100), nullable=False)
    customer_age: Mapped[int] = mapped_column(nullable=False)
    customer_gender: Mapped[str] = mapped_column(VARCHAR(20), nullable=False)

    tickets: Mapped[List["Tickets"]] = relationship(back_populates="customer")

    def __repr__(self):
        return f"<Customer(id={self.customer_id}, name='{self.customer_name}', email='{self.customer_email}')>"

In [60]:
class Tickets(Base):
    __tablename__ = "tickets" 

    owner:Mapped[int] = mapped_column(ForeignKey("customers.customer_id"))
    ticket_id: Mapped[int] = mapped_column(primary_key=True, nullable=False) 
    ticket_type: Mapped[str] = mapped_column(VARCHAR(100), nullable=False) 
    ticket_subject: Mapped[str] = mapped_column(VARCHAR(200), nullable=False) 
    ticket_description: Mapped[str] = mapped_column(nullable=False) 
    ticket_status: Mapped[int] = mapped_column(VARCHAR(50), nullable=False) 
    resolution: Mapped[Optional[str]] = mapped_column(nullable=True) 
    ticket_priority: Mapped[Optional[str]] = mapped_column(VARCHAR(50), nullable=True) 
    ticket_channel: Mapped[Optional[str]] = mapped_column(VARCHAR(50), nullable=True) 
    product_purchased: Mapped[int] = mapped_column(VARCHAR(100), nullable=False) 
    date_of_purchase: Mapped[date] = mapped_column(nullable=False) 
    first_response_time: Mapped[Optional[datetime]] = mapped_column(nullable=True) 
    time_to_resolution: Mapped[Optional[datetime]] = mapped_column(nullable=True) 
    customer_satisfaction_rating: Mapped[Optional[int]] = mapped_column(nullable=True) 

    customer: Mapped["Customers"] = relationship(back_populates="tickets")

    def __repr__(self):
        return f"<Ticket(id={self.ticket_id}, subject='{self.ticket_subject}')>"

dropping the existing tables from previous tests using normal sqlalchemy

In [61]:
Base.metadata.drop_all(engine, checkfirst=True)

creating the engine and the tables 

In [62]:
Base.metadata.create_all(engine)

inserting all the data points from the csv file

In [65]:
df = pd.read_csv("../data/Cleaned_Data.csv")

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

unique_customers = df[["customer_name", "customer_email", "customer_age", "customer_gender"]].drop_duplicates(subset=["customer_email"]).copy()
unique_customers.to_sql("customers", engine, if_exists="append", index=False)
print("Customers table inserted!")

# Fetch generated customer IDs from lowercase table
db_customers = pd.read_sql("SELECT customer_id, customer_email FROM customers", engine)

# Merge back onto dataframe and rename customer_id -> owner
tickets_df = df.merge(db_customers, on="customer_email", how="left")
tickets_df = tickets_df.rename(columns={"customer_id": "owner"})

tickets_to_insert = tickets_df[[
    "ticket_id", "owner", "product_purchased", "date_of_purchase",
    "ticket_type", "ticket_subject", "ticket_description", "ticket_status",
    "resolution", "ticket_priority", "ticket_channel", "first_response_time",
    "time_to_resolution", "customer_satisfaction_rating"
]]

tickets_to_insert.to_sql("tickets", engine, if_exists="append", index=False)
print("Tickets table inserted!")

print("Data uploaded successfully!")

Customers table inserted!
Tickets table inserted!
Data uploaded successfully!


In [66]:
with Session(engine) as session:
    stmt = select(Customers).where(Customers.customer_id == 1)
    for i in session.scalars(stmt):
        for j in i.tickets:
            print(j)

<Ticket(id=1, subject='Product setup')>
